In [1]:
import numpy as np
from time import perf_counter
from tscglue import data_loader

dataset = "Worms"
X_train, y_train, X_test, y_test = data_loader.load_fold(dataset, 0)
print(X_train.shape)

(181, 1, 900)


In [2]:
# 1. Chronos2 (baseline)
from tscglue.models_tsfm import Chronos2Embedding
t = Chronos2Embedding()
t0 = perf_counter()
emb = t.transform(X_train)
print(f"1. Chronos2 (baseline): {emb.shape} ({perf_counter()-t0:.1f}s)")

2026-03-21 23:46:52.887094: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


[Chronos2Embedding.transform] start X.shape=(181, 1, 900)
[Chronos2Embedding] torch.get_num_threads()=24, torch.get_num_interop_threads()=24, OMP_NUM_THREADS=unset, MKL_NUM_THREADS=unset
[Chronos2Embedding] from_pretrained(amazon/chronos-2) took 1.0s
[Chronos2Embedding._embed] batch 0: prep=0.00s embed=5.79s total=5.80s
[Chronos2Embedding._embed] pipeline=1.01s total=6.81s shape=(181, 768)
[Chronos2Embedding.transform] embed done 6.82s
[Chronos2Embedding] torch.get_num_threads()=24, torch.get_num_interop_threads()=24, OMP_NUM_THREADS=unset, MKL_NUM_THREADS=unset
[Chronos2Embedding._embed] batch 0: prep=0.00s embed=5.31s total=5.32s
[Chronos2Embedding._embed] pipeline=0.00s total=5.32s shape=(181, 768)
[Chronos2Embedding.transform] diff_embed done 12.16s
1. Chronos2 (baseline): (181, 1536) (12.2s)


In [3]:
# 2. Hydra fit FIRST (torch.set_num_threads(16))
from aeon.transformations.collection.convolution_based import HydraTransformer
h = HydraTransformer(n_jobs=16, random_state=0)
t0 = perf_counter()
h.fit(X_train)
print(f"2. Hydra fit: ({perf_counter()-t0:.1f}s)")

2. Hydra fit: (0.0s)


In [4]:
# 3. Chronos2 (after Hydra only - expect fast ~10s)
t = Chronos2Embedding()
t0 = perf_counter()
emb = t.transform(X_train)
print(f"3. Chronos2 (after Hydra only): {emb.shape} ({perf_counter()-t0:.1f}s)")

[Chronos2Embedding.transform] start X.shape=(181, 1, 900)
[Chronos2Embedding] torch.get_num_threads()=16, torch.get_num_interop_threads()=24, OMP_NUM_THREADS=unset, MKL_NUM_THREADS=unset
[Chronos2Embedding] from_pretrained(amazon/chronos-2) took 0.9s
[Chronos2Embedding._embed] batch 0: prep=0.00s embed=4.83s total=4.83s
[Chronos2Embedding._embed] pipeline=0.94s total=5.77s shape=(181, 768)
[Chronos2Embedding.transform] embed done 5.79s
[Chronos2Embedding] torch.get_num_threads()=16, torch.get_num_interop_threads()=24, OMP_NUM_THREADS=unset, MKL_NUM_THREADS=unset
[Chronos2Embedding._embed] batch 0: prep=0.00s embed=4.72s total=4.72s
[Chronos2Embedding._embed] pipeline=0.00s total=4.73s shape=(181, 768)
[Chronos2Embedding.transform] diff_embed done 10.53s
3. Chronos2 (after Hydra only): (181, 1536) (10.5s)


In [5]:
# 4. RDST fit (after Hydra - numba OpenMP will conflict with torch's pool)
from aeon.transformations.collection.shapelet_based import RandomDilatedShapeletTransform
rdst = RandomDilatedShapeletTransform(n_jobs=16, random_state=0)
t0 = perf_counter()
rdst.fit(X_train)
print(f"4. RDST fit: ({perf_counter()-t0:.1f}s)")

4. RDST fit: (2.3s)


In [6]:
# 5. Chronos2 (after Hydra+RDST - expect SLOW if theory correct)
t = Chronos2Embedding()
t0 = perf_counter()
emb = t.transform(X_train)
print(f"5. Chronos2 (after Hydra+RDST): {emb.shape} ({perf_counter()-t0:.1f}s)")

[Chronos2Embedding.transform] start X.shape=(181, 1, 900)
[Chronos2Embedding] torch.get_num_threads()=48, torch.get_num_interop_threads()=24, OMP_NUM_THREADS=unset, MKL_NUM_THREADS=unset
[Chronos2Embedding] from_pretrained(amazon/chronos-2) took 0.9s
[Chronos2Embedding._embed] batch 0: prep=0.00s embed=186.23s total=186.23s
[Chronos2Embedding._embed] pipeline=0.91s total=187.15s shape=(181, 768)
[Chronos2Embedding.transform] embed done 187.16s
[Chronos2Embedding] torch.get_num_threads()=48, torch.get_num_interop_threads()=24, OMP_NUM_THREADS=unset, MKL_NUM_THREADS=unset
[Chronos2Embedding._embed] batch 0: prep=0.00s embed=185.76s total=185.76s
[Chronos2Embedding._embed] pipeline=0.00s total=185.76s shape=(181, 768)
[Chronos2Embedding.transform] diff_embed done 372.95s
5. Chronos2 (after Hydra+RDST): (181, 1536) (372.9s)


In [7]:
# 6. Set threads to 48 (match what RDST set) - does matching fix it?
import torch
torch.set_num_threads(48)
print(f"Set torch threads to {torch.get_num_threads()}")
t = Chronos2Embedding()
t0 = perf_counter()
emb = t.transform(X_train)
print(f"6. Chronos2 (threads forced to 48): {emb.shape} ({perf_counter()-t0:.1f}s)")

Set torch threads to 48
[Chronos2Embedding.transform] start X.shape=(181, 1, 900)
[Chronos2Embedding] torch.get_num_threads()=48, torch.get_num_interop_threads()=24, OMP_NUM_THREADS=unset, MKL_NUM_THREADS=unset
[Chronos2Embedding] from_pretrained(amazon/chronos-2) took 1.0s
[Chronos2Embedding._embed] batch 0: prep=0.00s embed=107.31s total=107.32s
[Chronos2Embedding._embed] pipeline=0.99s total=108.31s shape=(181, 768)
[Chronos2Embedding.transform] embed done 108.33s
[Chronos2Embedding] torch.get_num_threads()=48, torch.get_num_interop_threads()=24, OMP_NUM_THREADS=unset, MKL_NUM_THREADS=unset
[Chronos2Embedding._embed] batch 0: prep=0.00s embed=106.34s total=106.34s
[Chronos2Embedding._embed] pipeline=0.00s total=106.34s shape=(181, 768)
[Chronos2Embedding.transform] diff_embed done 214.69s
6. Chronos2 (threads forced to 48): (181, 1536) (214.7s)
